# 💾 Download Images under 20minutes instead of 4hours

As the original script is not optimized, I have updated it to speed up the download from ~4hours to 20 minutes. 
Below you can find the approach ffor directl use on Kaggle and code snippet ready for copy paste into your project.

**Authors:**
- Eric Orenstein (eorenstein@mbari.org)
- Lukas Picek (lukaspicek@gmail.com)


# 🏃 Run this to retrive the images on Kaggle


⚠️ There is not enough space on Kaggle to store the images in original resolution. In that case, you have to resize it before saving. Just set `resize = True` ⚠️

In [4]:
import os
import json
import time
import logging
import requests

from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def make_session():
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    adapter = HTTPAdapter(max_retries=retry, pool_connections=32, pool_maxsize=32)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def resize_image(file_name):
    try:
        img = Image.open(file_name)
        img.thumbnail((1280, 720), Image.LANCZOS)
        img.save(file_name)
    except Exception as e:
        print(f"Problem resizing {file_name}: {e}")


import os
import requests
from PIL import Image, UnidentifiedImageError
from shutil import copyfileobj

def download_img(args, resize=True, timeout=30):
    name, url, outdir = args
    file_name = os.path.join(outdir, name)

    if os.path.exists(file_name):
        return 0

    try:
        with requests.get(url, stream=True, timeout=timeout) as resp:
            resp.raise_for_status()

            content_type = resp.headers.get("Content-Type", "").lower()
            if "image" not in content_type:
                print(f"Skipping non-image response for {name}: {content_type} | {url}")
                return 0

            tmp_name = file_name + ".part"
            with open(tmp_name, "wb") as f:
                copyfileobj(resp.raw, f)

        # verify image before finalizing
        try:
            with Image.open(tmp_name) as img:
                img.verify()
        except (UnidentifiedImageError, OSError) as e:
            print(f"Corrupt download for {name}: {e}")
            os.remove(tmp_name)
            return 0

        # reopen for optional resize
        if resize:
            try:
                with Image.open(tmp_name) as img:
                    img = img.convert("RGB")
                    img.thumbnail((1280, 720), Image.LANCZOS)
                    img.save(file_name)
                os.remove(tmp_name)
            except Exception as e:
                print(f"Resize failed for {name}: {e}")
                if os.path.exists(tmp_name):
                    os.remove(tmp_name)
                return 0
        else:
            os.replace(tmp_name, file_name)

        return 1

    except requests.RequestException as e:
        print(f"HTTP error for {name}: {e} | {url}")
        return 0
    except Exception as e:
        print(f"Unexpected error for {name}: {e} | {url}")
        return 0

def download_imgs(imgs, outdir=None, max_workers=16, resize=True):
    if outdir is None:
        outdir = os.path.join(os.getcwd(), "images")

    os.makedirs(outdir, exist_ok=True)

    args_list = [(name, url, outdir, resize) for name, url in imgs]

    downloaded = 0
    failed = 0
    existed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(download_img, args) for args in args_list]

        with tqdm(total=len(futures), desc="Downloading") as pbar:
            for future in as_completed(futures):
                status, info = future.result()

                if status == "downloaded":
                    downloaded += 1
                elif status == "exists":
                    existed += 1
                else:
                    failed += 1
                    print(f"Failed: {info}")

                pbar.update(1)
                pbar.set_postfix(downloaded=downloaded, exists=existed, failed=failed)

    print(f"\nDone. downloaded={downloaded}, exists={existed}, failed={failed}")

In [5]:
from pathlib import Path

def inspect_file(path):
    path = Path(path)
    size = path.stat().st_size
    with open(path, "rb") as f:
        head = f.read(32)
    return size, head

root = Path("kaggle")
for p in list(root.rglob("*.png"))[:20]:
    size, head = inspect_file(p)
    print(p, "| size:", size, "| head:", head[:16])

kaggle\training-data\0e98ec0e-c1ce-4540-a0d4-d7ab82309dde.png | size: 0 | head: b''
kaggle\training-data\212db4d6-450d-404f-8973-1c911b5e1171.png | size: 730604 | head: b'<!DOCTYPE html>\n'
kaggle\training-data\26f7878f-ebe8-42b0-a6a7-12241e1d71bc.png | size: 730687 | head: b'<!DOCTYPE html>\n'
kaggle\training-data\51d854e9-af94-43c3-b40c-f90c76e67327.png | size: 730547 | head: b'<!DOCTYPE html>\n'
kaggle\training-data\60833324-6275-49cd-869e-a727b80c16f6.png | size: 730539 | head: b'<!DOCTYPE html>\n'
kaggle\training-data\691bbe67-06fd-498b-a721-05181356da59.png | size: 730373 | head: b'<!DOCTYPE html>\n'
kaggle\training-data\754e6a28-a8eb-4cb3-a0b9-3f2d5daacbae.png | size: 730388 | head: b'<!DOCTYPE html>\n'
kaggle\training-data\bdeabd8b-92a1-4935-b890-e5c11f5b37d6.png | size: 730438 | head: b'<!DOCTYPE html>\n'
kaggle\training-data\dbc11dc2-91b5-4742-8318-f1805717a715.png | size: 730607 | head: b'<!DOCTYPE html>\n'


In [6]:
dataset = "train.json"
outpath = "kaggle/training-data"

logging.info(f"opening {dataset}")
with open(dataset, "r") as ff:
    dataset = json.load(ff)

ims = dataset["images"]
logging.info(f"retrieving {len(ims)} images")

ims = [(im["file_name"], im["coco_url"]) for im in ims]

download_imgs(ims, outdir=outpath, max_workers=16, resize=True)

Downloading:   0%|          | 0/8058 [00:00<?, ?it/s]


ValueError: too many values to unpack (expected 3)

In [ ]:
dataset = "eval.json"
outpath = "kaggle/test-data"

logging.info(f"opening {dataset}")
with open(dataset, "r") as ff:
    dataset = json.load(ff)

ims = dataset["images"]
logging.info(f"retrieving {len(ims)} images")

ims = [(im["file_name"], im["coco_url"]) for im in ims]

download_imgs(ims, outdir=outpath, max_workers=16, resize=True)